In [0]:
from pyspark.sql import functions as F

### master join function

In [0]:
products_path = "/Volumes/workspace/default/olist_files/olist_products_dataset.csv"
silver_products_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(products_path) \
    .select("product_id", "product_category_name", "product_weight_g") \
    .filter(F.col("product_category_name").isNotNull())

In [0]:
display(silver_products_df.limit(10))

In [0]:
translation_path = "/Volumes/workspace/default/olist_files/product_category_name_translation.csv"

translation_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(translation_path)

In [0]:
display(translation_df.limit(10))

### learn left join in pyspark

In [0]:

joined_df = silver_products_df.join(
    translation_df, 
    on="product_category_name", 
    how="left"
)

In [0]:
final_products_df = joined_df.withColumn(
    "product_category_name_english",
    F.coalesce(
        F.col("product_category_name_english"),
        F.lit("Unknown")
        )
)

print(f"名称空值的个数为{final_products_df \
    .filter(F.col('product_category_name_english').isNull()) \
    .count()}")

### learn how to use groupBy() function and .sort() function

In [0]:
# calculate the number of products in each category
report_df = final_products_df \
    .groupBy("product_category_name_english") \
    .agg(F.count("product_id").alias("total_products")) \
    .sort(F.col("total_products").desc())

# display the top 10
display(report_df.limit(10))


### study how to count duplicate values

step1: dataset.count()

step2: dataset.select("field name").distinct().count()



In [0]:
orders_path = "/Volumes/workspace/default/olist_files/olist_order_items_dataset.csv"

order_items_df = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(orders_path)


total_count = order_items_df.count()
unique_order_ids = order_items_df.select("order_id").distinct().count()

print(f"There are {total_count} rows in the order_items dataset and {unique_order_ids} are unique order IDs")

### study how to remove duplicate values

.dropDuplicates(['filed name'])

In [0]:
order_items_clean_df = order_items_df \
                            .filter(F.col("order_item_id").isNotNull()) \
                            .dropDuplicates(["order_id"])

total_count_clean = order_items_clean_df.count()
unique_order_ids_clean = order_items_clean_df.select("order_id").distinct().count()

print(f"There are {total_count_clean} rows in the order_items dataset after cleaning and {unique_order_ids_clean} are unique order IDs")